# FreeSurfer Longitudinal Processing Pipeline Using VIP Platform

## Overview

This repository contains scripts and instructions to run **FreeSurfer longitudinal pipeline** encompassing three processing steps (cross, base, long) using **VIP Client** and **Girder** for data storage and retrieval. The pipeline supports **3D T1-weighted MRIs** for whole brain segmentation.

The process is segmented in the following steps :
   1. **Download MRI Data from Girder**
   1. **Run FreeSurfer Longitudinal [CROSS]**: run the standard cross-sectional `recon-all` pipeline
   1. **Group Per-Subject Timepoints**: these will be used as inputs for step 4
   1. **Run FreeSurfer Longitudinal [BASE]** : run `recon-all -base` and create within subject template
   1. **Run FreeSurfer Longitudinal [LONG]**: run `recon-all -long` and create subject-specific longitudinal segmentations.
   1. **Upload FreeSurfer Outputs Back to Girder**: this step re-upload the `directives/freesurfer` folder containing the outputs back to Girder.
  1. **Cleanup**

The pipeline is fully automated with Python scripts, integrates VIP job submission, monitoring, and local download, and ensures all derivatives are centralized on Girder.

⚠️ Please **DO NOT** run the full notebook at once, but proceed cell by cell to ensure the inputs and outputs are correct after each step, this will allow you to control your execution and revert changes if needed.

⚠️ If you are closing the notebook (pipelines running in background or pausing between steps), remember to run the Libraries and Parameters cells (before step 1) again before continuing your work.

## Requirements

### Software

- Python ≥ 3.11
- `vip-client` Python package
- `girder_client` Python package

### VIP

- VIP account and API key
- Access to the VIP **FreeSurfer pipelines**:
    - `FreeSurfer-Recon-all/7.3.1`
    - `FreeSurfer-Recon-all-BASE/7.3.1`
    - `FreeSurfer-Recon-all-LONG/7.3.1`

### Girder

- Girder account and API key

### Data

- BIDS-compliant T1-weighted MRI dataset:
```
bids/
├── sub-*/
│   └── ses-*/
│       └── anat/
│           └── *_T1w.nii.gz
└── directives/
    └── freesurfer/
```
- FreeSurfer license file

This setup allows **fully automated longitudinal FreeSurfer processing** with reproducible outputs stored both locally and on Girder for sharing or further analysis.

## Libraries and functions

In [ ]:
# Built-ins
import tarfile
import shutil
from pathlib import Path

# VIP
from vip_client import VipSession

def flatten_folder(folder):
    parent = folder.parent
    for f in folder.iterdir():
        f.rename(parent / f.name)
    shutil.rmtree(folder)


## Parameters

**User variables**: should be checked and supplied at each execution. This cell will create a user_config.py file. Please modify the variable values by editing the user_config.py file after it is created.

⚠️ We strongly advise you not to modify the values directly in this cell in order to avoid conflicts when updating the notebook.

In [ ]:
CONFIG_PATH = Path('user_config.py')

user_config = f"""
import os
from pathlib import Path
from girder_client import GirderClient

# API keys
VIP_KEY = 'VIP_API_KEY' # Your VIP API key (string, file or environment variable with os.environ['key'])
GIRDER_KEY = 'GIRDER_API_KEY' # Your Girder API key (string or environment variable with os.environ['key'])

# Girder
GIRDER_CLIENT = GirderClient(apiUrl='GIRDER_URL') # Your Girder API endpoint url (ex : https://srmnopt.creatis.insa-lyon.fr/warehouse/api/v1 or https://myriad.creatis.insa-lyon.fr/api/v1)
GIRDER_DATASET_PATH = '/collection/COLLECTION_NAME/FOLDER_NAME' # Your Girder path to the dataset, can be in collections or user folders
GIRDER_OUTPUT_PATH = '/user/USER_NAME/FOLDER_NAME' # Your Girder path to the final outputs folder (must already exist), can be in collections or user folders

# Local
LOCAL_DATASET_PATH = Path('~/PATH').expanduser() # Your local dataset path, where the Girder dataset will be downloaded, must already exist
LICENSE_PATH = Path('~/PATH/LICENSE_NAME.txt').expanduser() # Your pipeline license path, will be copied into VIP input_dirs automatically
"""

if CONFIG_PATH.exists():
    answer = input("The user_config.py file already exists. Overwrite config ? (y/N): ")
    if answer != "y":
        print("File not created, as user_config.py already exists.")
    else:
        CONFIG_PATH.write_text(user_config)
        print("User_config.py overwritten with default values.")
else:
    CONFIG_PATH.write_text(user_config)
    print("User_config.py created with default values.")


Once you have filled the user_config.py file with your custom parameters, you can run the following cell to load the configuration.

In [ ]:
import importlib
import user_config as cfg

# Reload user config
importlib.reload(cfg)

**Constant variables**: should not be changed unless the dataset structure or the pipeline have been modified


In [ ]:
# Name of the pipelines
PIPELINE_ALL_ID = 'FreeSurfer-Recon-all/7.3.1'
PIPELINE_BASE_ID = 'FreeSurfer-Recon-all-BASE/7.3.1'
PIPELINE_LONG_ID = 'FreeSurfer-Recon-all-LONG/7.3.1'

# Path to the FreeSurfer (output) folder
FREESURFER_DIR = Path(cfg.LOCAL_DATASET_PATH / 'derivatives' / 'freesurfer')

# FreeSurfer CROSS
FS_CROSS_ID = 'CROSS'
FS_CROSS_OUTPUT_DIR = Path(FREESURFER_DIR / FS_CROSS_ID / 'Outputs')

# FreeSurfer BASE
FS_BASE_ID = 'BASE'
FS_BASE_INPUT_DIR = Path(FREESURFER_DIR / FS_BASE_ID / 'Inputs')
FS_BASE_OUTPUT_DIR = Path(FREESURFER_DIR / FS_BASE_ID / 'Outputs')

# FreeSurfer LONG
FS_LONG_ID = 'LONG'
FS_LONG_OUTPUT_DIR = Path(FREESURFER_DIR / FS_LONG_ID / 'Outputs')

# VIP
SESSION_DATA_NAME = 'session_data.json'
SESSIONS = {FS_CROSS_ID : FS_CROSS_OUTPUT_DIR,
            FS_BASE_ID : FS_BASE_OUTPUT_DIR,
            FS_LONG_ID : FS_LONG_OUTPUT_DIR}

LICENSES = {FS_CROSS_ID : cfg.LOCAL_DATASET_PATH / f'license_{FS_CROSS_ID}.txt',
            FS_BASE_ID : FS_BASE_INPUT_DIR / f'license_{FS_BASE_ID}.txt',
            FS_LONG_ID : FS_BASE_OUTPUT_DIR / f'license_{FS_LONG_ID}.txt'}

## 1. Download MRI Data from Girder

This script authenticates to the Girder warehouse, downloads the input BIDS-compliant MRI dataset locally, and ensure the FreeSurfer derivatives directory structure exists.

### ⚙️ Inputs

- `GIRDER_DATASET_PATH`: Path of the Girder folder containing MRI data
- `LOCAL_DATASET_PATH`: Local path where the data will be downloaded

### 📤 Outputs

- Local copy of the MRI dataset
- Created directory: `derivatives/freesurfer/` inside the download folder if not already present.

In [ ]:
# Authentication to Girder
cfg.GIRDER_CLIENT.authenticate(apiKey=cfg.GIRDER_KEY)

# Download dataset from Girder
cfg.GIRDER_CLIENT.downloadFolderRecursive(cfg.GIRDER_DATASET_PATH, str(cfg.LOCAL_DATASET_PATH))

# Create LOCAL_DATASET_PATH, derivatives and freesurfer folders if they don't exist
FREESURFER_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Done: 'derivatives/freesurfer' folder ensured at: {FREESURFER_DIR}, Girder dataset copied to : {cfg.LOCAL_DATASET_PATH}")

## 2. Run FreeSurfer Longitudinal [CROSS]

This script launches the **standard FreeSurfer** `recon-all` **cross sectional** pipeline on VIP for all T1-weighted MRIs and download the results locally. The script:

- Finds all `*_T1w.nii.gz` files in `sub-*` folders
- Keeps only the `run-01` scans (ignores additional runs)
- Builds subject IDs automatically from the filenames (removing `.nii.gz`)
- Submits parallel jobs to the VIP FreeSurfer recon-all pipeline
- Downloads the completed outputs locally to `derivatives/freesurfer/CROSS/Outputs`

### File Selection Rules

**Note**: If two or more T1-weighted sequences were present at the same session, they were labeled `*_run-01_T1w` and `_run-02_T1w`. The **non-gadolinium (no Gado)** scan (if present) was labeled as `*_run-01_T1w`, as it is preferred for segmentation.

So, the script uses only files ending with:

- `*_T1w.nii.gz` (no run specification) OR
- `*_run-01_T1w.nii.gz`

⚠️ If processing of `*_run-01_T1w.nii.gz` fails, you can try `*_run-02_T1w.nii.gz` or `*_run-03_T1w.nii.gz` directly on the platform. Make sure to put the results back in the derivatives/freesurfer folder.

### ⚙️ Inputs

- `nifti`: List of T1-weighted NIfTI files (`*_T1w.nii.gz`) from `sub-*` folders, keeping only `run-01`
- `license`: FreeSurfer license file
- `subjid`: automatically generated subject IDs used to name output folders for each NIfTI file (derived from the filename minus `.nii.gz`)

### 📤 Outputs

- FreeSurfer cross-sectional results in `derivatives/freesurfer/CROSS/Outputs`

⚠️ Review the outputs carefully and note which results to keep or exclude/re-run before proceeding to the next steps.

In [ ]:
# Remove FreeSurfer CROSS output directory if existing
if FS_CROSS_OUTPUT_DIR.exists():
    shutil.rmtree(FS_CROSS_OUTPUT_DIR)

# Create FreeSurfer CROSS output directory
FS_CROSS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Get T1w NIfTIs only sub-* folders, excluding run-02, run-03, ...
nifti_files = [
    str(f)
    for sub in cfg.LOCAL_DATASET_PATH.iterdir()
    if sub.is_dir() and sub.name.startswith('sub-')
    for f in sub.rglob('*.nii.gz')
    if f.name.endswith('_T1w.nii.gz')
       and ('_run-' not in f.name or '_run-01_' in f.name)
]

# Create subjid from filenames
subjid = [Path(f).name.replace('.nii.gz', '') for f in nifti_files]

input_settings = {
    'nifti': nifti_files,
    'license': shutil.copy2(cfg.LICENSE_PATH, LICENSES[FS_CROSS_ID]),
    'subjid': subjid,
}

session = VipSession.init(
    api_key=cfg.VIP_KEY,
    input_dir=str(cfg.LOCAL_DATASET_PATH),
    output_dir= str(FS_CROSS_OUTPUT_DIR),
    pipeline_id=PIPELINE_ALL_ID,
    input_settings=input_settings
)

session.upload_inputs()
session.launch_pipeline()
session.monitor_workflows()
session.display()

Run this cell after the execution ended to download outputs and setup output directory

In [ ]:
# Connect back to session if needed
VipSession.init(api_key=cfg.VIP_KEY)
session = VipSession(output_dir=str(FS_CROSS_OUTPUT_DIR))

# Download outputs to the output_dir
session.download_outputs(False, get_status=['Finished', 'Killed'])

# Since results may arrive in a VipSession folder (2026-X), when it's the case we flatten this folder to put files directly under FS_CROSS_OUTPUT_DIR
for f in FS_CROSS_OUTPUT_DIR.iterdir():
    if f.is_dir():
        flatten_folder(Path(f))

## 3. Group Per-Subject Timepoints

This script prepares the **subject-specific timepoints tarballs for the BASE step** by grouping cross-sectional timepoints (`ses-*`) for each subject into a single archive. Each archive contains all sessions for one subject, ready for the `recon-all -base` pipeline. The script:

- Detects all eligible cross-sectional tarballs in `derivatives/freesurfer/CROSS/Outputs` :
    - Must contain `ses-`
    - Must **not** contain `.long.`
- Extracts each archive temporarily
- Groups timepoints by subject (`sub-XXXX`)
- Re-compresses them into one `.tgz` per subject
- Cleans temporary extracted files

### ⚙️ Inputs

- `FS_CROSS_OUTPUT_DIR`: Path to the `derivatives/freesurfer/Cross/Outputs` directory containing cross-sectional `.tar.gz` or `.tgz` outputs from downloaded from VIP in step 1

### 📤 Outputs

- Grouped per-subject archives for longitudinal processing: `derivatives/freesurfer/BASE/Inputs`

In [ ]:
# Remove FreeSurfer BASE inputs/outputs directories if existing
if FS_BASE_INPUT_DIR.exists():
    shutil.rmtree(FS_BASE_INPUT_DIR)
if FS_BASE_OUTPUT_DIR.exists():
    shutil.rmtree(FS_BASE_OUTPUT_DIR)

# Create FreeSurfer BASE inputs/outputs directories
FS_BASE_INPUT_DIR.mkdir(parents=True, exist_ok=True)
FS_BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Collect eligible tarballs (.tar.gz or .tgz)
tarballs = [
    t for t in FS_CROSS_OUTPUT_DIR.iterdir()
    if t.is_file()
        and t.suffixes in ([".tar", ".gz"], [".tgz",])
        and "ses-" in t.name
        and ".long." not in t.name
]

subjects = {}
for tar_path in tarballs:
    subj = tar_path.name.split("_")[0]  # sub-XXXX

    # Remove full archive suffixes safely
    base_name = tar_path.name[:-(sum(len(s) for s in tar_path.suffixes))]

    extract_dir = FS_BASE_INPUT_DIR / "extracted" / base_name
    extract_dir.mkdir(parents=True, exist_ok=True)

    # Untar (auto-detect compression)
    with tarfile.open(tar_path, "r:*") as tar:
        tar.extractall(path=extract_dir)

    subjects.setdefault(subj, []).append(extract_dir)

# Group per subject
for subj, tp_dirs in subjects.items():
    out_tar = FS_BASE_INPUT_DIR / f"{subj}_TPs.tgz"
    with tarfile.open(out_tar, "w:gz") as tar:
        for tp_dir in tp_dirs:
            for item in tp_dir.iterdir():
                tar.add(item, arcname=item.name)

# Cleanup extracted files
shutil.rmtree(FS_BASE_INPUT_DIR / "extracted", ignore_errors=True)

print(f"✅ Done: grouped longitudinal timepoints per subject at '{FS_BASE_INPUT_DIR}'.")

## 4. Run FreeSurfer Longitudinal [BASE]

This script runs **FreeSurfer** `recon-all -base` on VIP using the grouped timepoints from Step 3. It generates the **within-subject template** required for the following longitudinal processing stream.

The script:

- Finds all grouped archives (`*_TPs.tgz`) in the `BASE/Inputs` folder
- Extracts BASE IDs for each subject
- Launches the VIP **recon-all -base** pipeline
- Downloads the completed BASE outputs locally

### ⚙️ Inputs

- `TP_TARBALL`: List of grouped timepoint archives (`*_TPs.tgz`) for each subject
- `LICENSE_FILE`: FreeSurfer license file
- `BASE_ID`: Object IDs corresponding to each BASE template, automatically extracted from the tarball filenames and used to name the output folders for each subject

### 📤 Outputs

- Outputs will be downloaded to: `derivatives/freesurfer/BASE/Outputs`

In [ ]:
# Collect tarballs and BASE_IDs
tp_tarballs = [f for f in FS_BASE_INPUT_DIR.iterdir() if f.suffix in (".tgz", ".tar.gz") and "_TPs" in f.name]
base_ids = [f.name.split("_")[0] for f in tp_tarballs]

# Build batch input settings
input_settings = {
    "LICENSE_FILE": shutil.copy2(cfg.LICENSE_PATH, LICENSES[FS_BASE_ID]),
    "TP_TARBALL": [str(f) for f in tp_tarballs],  # list of tarballs
    "BASE_ID": base_ids,                          # list of base IDs
}

session = VipSession.init(
    api_key=cfg.VIP_KEY,
    input_dir=str(FS_BASE_INPUT_DIR),
    output_dir= str(FS_BASE_OUTPUT_DIR),
    pipeline_id=PIPELINE_BASE_ID,
    input_settings=input_settings
)

session.upload_inputs()
session.launch_pipeline() # VIP will process tarballs in parallel
session.monitor_workflows()
session.display()

Run this cell after the execution ended to download outputs and setup output directory

In [ ]:
# Connect back to session if needed
VipSession.init(api_key=cfg.VIP_KEY)
session = VipSession(output_dir=str(FS_BASE_OUTPUT_DIR))

# Download outputs to the output_dir
session.download_outputs(False, get_status=['Finished', 'Killed'])

# Since results may arrive in a VipSession folder (2026-X), when it's the case we flatten this folder to put files directly under FS_BASE_OUTPUT_DIR
for f in FS_BASE_OUTPUT_DIR.iterdir():
    if f.is_dir():
        flatten_folder(Path(f))

## 5. Run FreeSurfer Longitudinal [LONG]

This script runs **FreeSurfer** `recon-all -long` on VIP using the BASE templates from Step 4 and the timepoints (TPs) tarballs produced from Step 3 and were used as an input to step 4, producing segmentations more robustly by registering each timepoint to its corresponding subject template. The script:

- Connects to VIP using the API key
- Lists BASE templates and TP tarballs on VIP
- Matches BASE templates with corresponding TP tarballs for each subject
- Prepares VIP input settings for the LONG pipeline
- Launches `recon-all -long` jobs in parallel on VIP
- Monitors workflow progress
- Downloads the completed longitudinal outputs to the local output directory

### ⚙️ Inputs

- `TP_TARBALL`: List of grouped timepoint archives (`*_TPs.tgz`) for each subject (BASE inputs)
- `LICENSE_FILE`: FreeSurfer license file
- `BASE_ID`: Subject-specific BASE template folder (BASE outputs)

### 📤 Outputs

- Outputs will be downloaded to: `derivatives/freesurfer/LONG/Outputs`
- Output folders will be named following this pattern: `sub-*_ses-*_T1w.long.sub-*.`

In [ ]:
# Remove FreeSurfer LONG output directory if existing
if FS_LONG_OUTPUT_DIR.exists():
    shutil.rmtree(FS_LONG_OUTPUT_DIR)

# Create FreeSurfer LONG output directory
FS_LONG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# List BASE and TP files
base_files = [
    f
    for f in FS_BASE_OUTPUT_DIR.iterdir()
    if f.is_file() and f.name.startswith("sub-") and "_TPs" not in f.name
]

tp_files = [
    f
    for f in FS_BASE_INPUT_DIR.iterdir()
    if f.is_file() and f.name.startswith("sub-") and "_TPs" in f.name
]

# Match BASE → TP tarball
base_dict = {f.name.split(".")[0]: str(f) for f in base_files}
tp_dict   = {f.name.split("_TPs")[0]: str(f) for f in tp_files}
subjects = set(base_dict.keys()) & set(tp_dict.keys())
if not subjects:
    raise RuntimeError("No matching BASE and TP_TARBALL found!")

# Input settings for the LONG pipeline
input_settings = {
    "LICENSE_FILE": shutil.copy2(cfg.LICENSE_PATH, LICENSES[FS_LONG_ID]),
    "BASE_ID": [base_dict[sub] for sub in subjects],
    "TP_TARBALL": [tp_dict[sub] for sub in subjects],
    "directives": "-all",
}

print(f"Submitting {len(subjects)} LONG jobs for subjects: {', '.join(subjects)}")

session = VipSession.init(
    api_key=cfg.VIP_KEY,
    input_dir=str(FS_BASE_OUTPUT_DIR.parent),  # LONG pipeline uses data from BASE inputs and outputs
    output_dir=str(FS_LONG_OUTPUT_DIR),
    pipeline_id=PIPELINE_LONG_ID,
    input_settings=input_settings
)

session.upload_inputs()
session.launch_pipeline()
session.monitor_workflows()
session.display()

Run this cell after the execution ended to download outputs and setup output directory

In [ ]:
# Connect back to session if needed
VipSession.init(api_key=cfg.VIP_KEY)
session = VipSession(output_dir=str(FS_LONG_OUTPUT_DIR))

# Download outputs to the output_dir
session.download_outputs(False, get_status=['Finished', 'Killed'])

# Since results may arrive in a VipSession folder (2026-X), when it's the case we flatten this folder to put files directly under FS_LONG_OUTPUT_DIR
for f in FS_LONG_OUTPUT_DIR.iterdir():
    if f.is_dir():
        flatten_folder(Path(f))

## 6. Upload FreeSurfer outputs back to Girder

This script uploads **local FreeSurfer outputs** (from CROSS, BASE, and LONG pipelines) back to the **Girder warehouse** for centralized storage and sharing. The script:

- Uploads the local FreeSurfer `derivatives/freesurfer` back to Girder, the user needs to have write access to target Girder path
- Temporarily moves session_data.json files and licenses so they are not uploaded on Girder

### ⚙️ Inputs

- Local FreeSurfer derivatives folder `FREESURFER_DIR`
- Girder target folder path `GIRDER_OUTPUT_PATH`

### 📤 Outputs

- Uploads all FreeSurfer derivative files (`derivatives/freesurfer/`) to the specified Girder folder

In [ ]:
# Restore all session_data.json and licenses files
def restore_files(files):
    for original_path, temp_path in files:
        if temp_path.exists():
            shutil.move(str(temp_path), str(original_path))

# Authenticate back to Girder
cfg.GIRDER_CLIENT.authenticate(apiKey=cfg.GIRDER_KEY)

moved_files = []
for session_id in SESSIONS:
    session_data_path = SESSIONS[session_id] / SESSION_DATA_NAME
    license_path = LICENSES[session_id]
    if session_data_path.exists():
        # Move VIP session_data jsons so they are not uploaded on Girder
        temp_path = FREESURFER_DIR.parent / f'{session_id}_session_data.json'
        shutil.move(str(session_data_path), str(temp_path))
        moved_files.append((session_data_path, temp_path))
    if license_path.exists():
        # Move VIP pipelines licenses so they are not uploaded on Girder
        temp_path = FREESURFER_DIR.parent / f'license_{session_id}.txt'
        shutil.move(str(license_path), str(temp_path))
        moved_files.append((license_path, temp_path))

try :
    # Upload full freesurfer dir (outputs) to Girder selected path GIRDER_OUTPUT_PATH
    cfg.GIRDER_CLIENT.upload(str(FREESURFER_DIR), cfg.GIRDER_CLIENT.resourceLookup(cfg.GIRDER_OUTPUT_PATH)['_id'], leafFoldersAsItems=True, reuseExisting=True)
finally:
    restore_files(moved_files)
    print(f"✅ Done: uploaded FreeSurfer outputs to Girder path '{cfg.GIRDER_OUTPUT_PATH}'.")

## 7. CLEANUP

Run to close VIP sessions, deleting inputs and outputs of different steps on VIP. Also removes VIP licenses and session data locally.
All inputs and outputs can still be found locally and deleted when needed.

In [ ]:
for session_id in SESSIONS:
    try:
        # Cleanup on VIP, deletes inputs and ouputs
        VipSession(output_dir=str(SESSIONS[session_id])).finish()
    except:
        print(f"Error closing session {session_id} on VIP")
    finally:
        # Cleanup locally, deletes session_data jsons and licenses
        session_data_path = SESSIONS[session_id] / SESSION_DATA_NAME
        license_path = LICENSES[session_id]
        if session_data_path.exists():
            session_data_path.unlink()
        if license_path.exists():
            license_path.unlink()

print("✅ Done: cleaned up all used sessions on VIP, and removed session data and license files locally.")